# 01 — Factor Momentum Foundations

> Esta investigación pregunta si parte del momentum observado en activos puede entenderse como exposición a factores o grupos sistemáticos que también presentan persistencia temporal.

$ R_{i,t} = \sum_{k=1}^{K} \beta_{ik}F_{k,t} + \epsilon_{i,t} $

La primera etapa usará proxies ETF observables y líquidos, no factores académicos puros. La ecuación es un marco de investigación; no supone todavía que los proxies expliquen los retornos de los activos.


## <b>Relación con TSMOM</b>

`01_tsmom` ya establece un baseline de momentum temporal. Allí la pregunta es: ¿el activo viene subiendo o bajando respecto a su propia historia? Aquí la pregunta cambia a: ¿ese momentum coincide con momentum persistente en factores o grupos sistemáticos?

Esta línea complementa el baseline; no lo reemplaza ni reutiliza todavía abstracciones de código comunes.


## <b>Datos</b>

> Todo análisis posterior leerá exclusivamente `data/gold/daily_series/etf/<SYMBOL>.parquet`. Raw, Silver, `series/` e intradía quedan fuera del análisis.

El universo proxy inicial propuesto es SPY, QQQ, IWM, IWD, IWF, XLE y XLF. La disponibilidad se comprobará antes de calcular retornos.


## <b>Block 01 — Factor universe</b>

> Verificar el universo de proxies ETF disponible en Gold, su cobertura común y los límites de interpretación de cada proxy.

Estos ETFs son proxies económicos observables. En esta primera etapa permiten estudiar persistencia relativa entre estilos y sectores sin asumir todavía una estructura factorial académica exacta.


In [ ]:
from datetime import date
from pathlib import Path

import plotly.graph_objects as go
import polars as pl

PROXIES = {
    "SPY": "broad US equity market",
    "QQQ": "large-cap growth / technology tilt",
    "IWM": "small-cap tilt",
    "IWD": "value tilt",
    "IWF": "growth tilt",
    "XLE": "energy sector",
    "XLF": "financial sector",
}
RESEARCH_START = date(2018, 1, 2)
RESEARCH_END = date(2026, 11, 17)

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        for project_root in (candidate, candidate / "Momentum_systematic_strategies"):
            if (project_root / "data" / "gold" / "daily_series" / "etf").is_dir():
                return project_root
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf desde el directorio actual.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"


In [ ]:
frames = {}
audit_rows = []

for symbol, proxy_role in PROXIES.items():
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")

    frame = pl.read_parquet(path).with_columns(pl.col("session_date").cast(pl.Date))
    if "adjusted_close" not in frame.columns:
        raise ValueError(f"{symbol} no contiene adjusted_close.")

    frame = frame.filter(
        pl.col("session_date").is_between(RESEARCH_START, RESEARCH_END, closed="both")
    ).sort("session_date")
    dates = frame.get_column("session_date").to_list()
    relevant_nulls = int(frame.select(["session_date", "adjusted_close"]).null_count().sum_horizontal()[0])
    duplicates = int(frame.get_column("session_date").is_duplicated().sum())

    if not frame.height or relevant_nulls or duplicates or dates != sorted(dates):
        raise ValueError(f"{symbol} no supera los controles básicos de cobertura.")
    if bool((frame.get_column("adjusted_close") <= 0).any()):
        raise ValueError(f"{symbol} contiene adjusted_close no positivo.")

    frames[symbol] = frame.select("session_date", "adjusted_close")
    audit_rows.append({
        "symbol": symbol,
        "proxy_role": proxy_role,
        "start_date": frame.get_column("session_date").min(),
        "end_date": frame.get_column("session_date").max(),
        "n_sessions": frame.height,
        "nulls": relevant_nulls,
        "duplicates": duplicates,
    })

universe_audit = pl.DataFrame(audit_rows)
universe_audit


La tabla confirma `adjusted_close`, precios positivos, fechas ordenadas, ausencia de duplicados y nulos relevantes. El control de calidad detallado ya corresponde a la promoción Gold.


In [ ]:
common_panel = frames["SPY"].rename({"session_date": "date", "adjusted_close": "SPY"})
for symbol in list(PROXIES)[1:]:
    common_panel = common_panel.join(
        frames[symbol].rename({"session_date": "date", "adjusted_close": symbol}),
        on="date",
        how="inner",
    )

common_panel = common_panel.sort("date")
panel_summary = {
    "dimensions": common_panel.shape,
    "first_date": common_panel.get_column("date").min(),
    "last_date": common_panel.get_column("date").max(),
    "nulls": int(common_panel.null_count().sum_horizontal()[0]),
    "common_sessions": common_panel.height,
}
panel_summary


In [ ]:
normalized_prices = common_panel.select(
    "date",
    *[(pl.col(symbol) / pl.col(symbol).first()).alias(symbol) for symbol in PROXIES],
)

figure = go.Figure()
for symbol in PROXIES:
    figure.add_scatter(x=normalized_prices["date"], y=normalized_prices[symbol], mode="lines", name=symbol)

figure.update_layout(
    title="Factor proxy universe — normalized adjusted prices",
    yaxis_title="P_t / P_0",
    legend_title_text="",
)
figure.show()


### <b>Block 01 conclusion</b>

- Los siete proxies propuestos — SPY, QQQ, IWM, IWD, IWF, XLE y XLF — están disponibles en Gold.
- La ventana común exacta es `2018-01-02 → 2026-11-17`, con 2,232 sesiones.
- Cada serie conserva 2,232 observaciones, `adjusted_close` disponible, precios positivos y cero nulos o fechas duplicadas en la ventana.
- No quedan diferencias de cobertura entre los proxies dentro del rango de investigación.
- El panel `date | SPY | QQQ | IWM | IWD | IWF | XLE | XLF` queda preparado para construir factor returns.

El siguiente bloque estudiará los retornos diarios de estos proxies.


## <b>Block 02 — Factor returns</b>

> Los retornos se calculan sobre el panel Gold validado de Block 01. Son descripciones de proxies ETF, no retornos de factores académicos puros.

$ r_t = \ln(P_t / P_{t-1}) $


In [ ]:
return_columns = [f"{symbol}_ret" for symbol in PROXIES]
returns_panel = common_panel.with_columns([
    pl.col(symbol).log().diff().alias(f"{symbol}_ret") for symbol in PROXIES
]).drop_nulls(return_columns)

returns_panel.select("date", *return_columns).head()


In [ ]:
return_rows = []
for symbol in PROXIES:
    series = returns_panel.get_column(f"{symbol}_ret")
    return_rows.append({
        "symbol": symbol,
        "mean daily return (%)": series.mean() * 100,
        "annualized volatility (%)": series.std() * (252 ** 0.5) * 100,
        "min daily return (%)": series.min() * 100,
        "max daily return (%)": series.max() * 100,
        "valid observations": len(series),
    })

return_summary = pl.DataFrame(return_rows).with_columns(
    pl.col([
        "mean daily return (%)", "annualized volatility (%)",
        "min daily return (%)", "max daily return (%)",
    ]).round(3)
)
return_summary


### <b>Correlation structure</b>

Las correlaciones describen co-movimiento contemporáneo; no establecen causalidad ni sustituyen una exposición factorial estimada.


In [ ]:
correlation_table = returns_panel.select([
    pl.col(f"{symbol}_ret").alias(symbol) for symbol in PROXIES
]).corr().with_columns(pl.Series("proxy", list(PROXIES))).select("proxy", *PROXIES).with_columns(
    pl.all().exclude("proxy").round(3)
)

correlation_table


In [ ]:
figure = go.Figure(go.Heatmap(
    z=correlation_table.select(list(PROXIES)).to_numpy(),
    x=list(PROXIES),
    y=list(PROXIES),
    zmin=-1,
    zmax=1,
    colorscale="RdBu",
    zmid=0,
    texttemplate="%{z:.2f}",
))
figure.update_layout(title="Daily return correlation — factor proxy universe")
figure.show()


### <b>Relative return spreads</b>

> Los spreads son diferencias de retornos logarítmicos diarios entre proxies observables. Se usan para describir liderazgo relativo, no como factores académicos.


In [ ]:
SPREADS = {
    "IWF - IWD": ("IWF", "IWD"),
    "IWM - SPY": ("IWM", "SPY"),
    "QQQ - SPY": ("QQQ", "SPY"),
    "XLE - SPY": ("XLE", "SPY"),
    "XLF - SPY": ("XLF", "SPY"),
}

spread_columns = {label: f"spread_{long.lower()}_{short.lower()}" for label, (long, short) in SPREADS.items()}
spread_panel = returns_panel.with_columns([
    (pl.col(f"{long}_ret") - pl.col(f"{short}_ret")).alias(spread_columns[label])
    for label, (long, short) in SPREADS.items()
])

relative_performance = spread_panel.select(
    "date",
    *[(pl.col(column).cum_sum().exp()).alias(label) for label, column in spread_columns.items()],
)
relative_summary = pl.DataFrame([
    {
        "spread": label,
        "cumulative relative return (%)": (relative_performance.get_column(label)[-1] - 1) * 100,
    }
    for label in SPREADS
]).with_columns(pl.col("cumulative relative return (%)").round(3))
relative_summary


In [ ]:
figure = go.Figure()
for label in ("IWF - IWD", "IWM - SPY", "QQQ - SPY"):
    figure.add_scatter(x=relative_performance["date"], y=relative_performance[label], mode="lines", name=label)

figure.update_layout(
    title="Cumulative relative performance — selected proxy spreads",
    yaxis_title="relative wealth",
    legend_title_text="",
)
figure.show()


### <b>Rolling correlation and cross-sectional dispersion</b>

Las correlaciones móviles de 63 sesiones y la dispersión diaria muestran que el co-movimiento entre proxies puede cambiar con el tiempo.


In [ ]:
rolling_correlations = returns_panel.select(
    "date",
    *[
        pl.rolling_corr(pl.col("SPY_ret"), pl.col(f"{symbol}_ret"), window_size=63).alias(symbol)
        for symbol in ("QQQ", "IWM", "XLE", "XLF")
    ],
)

figure = go.Figure()
for symbol in ("QQQ", "IWM", "XLE", "XLF"):
    figure.add_scatter(x=rolling_correlations["date"], y=rolling_correlations[symbol], mode="lines", name=f"SPY / {symbol}")

figure.update_layout(
    title="SPY — 63-session rolling return correlation",
    yaxis_title="correlation",
    legend_title_text="",
)
figure.show()


In [ ]:
dispersion_panel = returns_panel.with_columns(
    pl.concat_list([pl.col(column) for column in return_columns]).list.std().alias("cross_sectional_dispersion")
)
dispersion = dispersion_panel.get_column("cross_sectional_dispersion")
dispersion_summary = pl.DataFrame([{
    "mean (%)": dispersion.mean() * 100,
    "median (%)": dispersion.median() * 100,
    "p90 (%)": dispersion.quantile(0.90) * 100,
    "max (%)": dispersion.max() * 100,
}]).with_columns(pl.all().round(3))
dispersion_summary


In [ ]:
figure = go.Figure()
figure.add_scatter(
    x=dispersion_panel["date"],
    y=dispersion_panel["cross_sectional_dispersion"] * 100,
    mode="lines",
    name="cross-sectional dispersion",
)
figure.update_layout(
    title="Daily cross-sectional dispersion — factor proxy returns",
    yaxis_title="standard deviation (%)",
    legend_title_text="",
)
figure.show()


### <b>Interpretación</b>

- QQQ e IWF son el par más correlacionado (0.983); SPY e IWF también se mueven de forma muy próxima (0.958).
- XLE muestra la mayor independencia relativa frente a QQQ (0.391) e IWF (0.423); IWM y XLF conservan co-movimiento relevante con el mercado amplio.
- Los acumulados muestran liderazgo relativo prolongado de growth frente a value: IWF - IWD termina en +40.4%; QQQ - SPY termina en +40.9%.
- Las correlaciones móviles cambian materialmente: SPY/XLE llega a ser negativa en algunos tramos y supera 0.9 en otros.
- La dispersión cross-sectional mediana es 0.574% diario, con p90 de 1.206%; hay heterogeneidad suficiente para estudiar momentum relativo, sin concluir todavía que exista factor momentum.


### <b>Block 02 conclusion</b>

- Se construyeron retornos logarítmicos diarios para los siete proxies sobre 2,231 sesiones comunes.
- La estructura de correlación combina pares muy cercanos, como QQQ/IWF, con comportamiento sectorial más diferenciado, especialmente XLE.
- Los spreads relativos y la dispersión muestran variación transversal, pero no son todavía señales ni resultados de una estrategia.
- No se estimaron betas, regresiones, rankings ni objetos de cartera.

Los retornos quedan preparados para estudiar persistencia temporal y relativa en Block 03.


## <b>Block 03 — Factor momentum</b>

> Describir la persistencia temporal de los retornos de los proxies sin construir aún señales ni estrategias.

_Placeholder — sin cálculos todavía._


## <b>Block 04 — Asset vs factor momentum</b>

> Comparar de forma descriptiva el momentum de activos con el de los proxies disponibles, sin estimar todavía betas ni regresiones factoriales.

_Placeholder — sin cálculos todavía._


## <b>Block 05 — Diagnostics</b>

> Revisar cobertura, concentración, correlaciones y límites de los proxies antes de profundizar en la relación entre activos y factores.

_Placeholder — sin cálculos todavía._
